# QE

In [4]:
from pymatgen.io.vasp import VolumetricData
from doped.analysis import DefectsParser
%matplotlib inline

In [5]:
bulk_path = "../tests/data/espresso/MgO_qe/Defects/MgO_bulk"
dielectric = 8.8963

In [6]:
calc_root = "../tests/data/espresso/MgO_qe/Defects"
pp_folder = "../tests/data/espresso/pp_folder"

beta_dp_dict = {}
for beta in [0.5, 1.0, 1.5, 2.0]:
    beta_dp_dict[beta] = DefectsParser(code = 'espresso',
                   output_path = calc_root,
                   dielectric= 8.8963,
                   beta = beta,
                   pp_folder = pp_folder, bulk_path = bulk_path, processes = 1)

Parsing Mg_O_+1/espresso_std: 100%|██████████| 4/4 [00:45<00:00, 11.49s/it]
[PARSED_WITH_WARNING] Folder: Mg_O_+3 | Warning: Projected magnetisation not implemented for QE. Returning None.

[PARSED_WITH_WARNING] Folder: Mg_O_+4 | Warning: Projected magnetisation not implemented for QE. Returning None.

[PARSED_WITH_WARNING] Folder: Mg_O_+2 | Warning: Projected magnetisation not implemented for QE. Returning None.

[PARSED_WITH_WARNING] Folder: Mg_O_+1 | Warning: Projected magnetisation not implemented for QE. Returning None.

Parsing Mg_O_+1/espresso_std: 100%|██████████| 4/4 [00:37<00:00,  9.32s/it]
[PARSED_WITH_WARNING] Folder: Mg_O_+3 | Warning: Projected magnetisation not implemented for QE. Returning None.

[PARSED_WITH_WARNING] Folder: Mg_O_+4 | Warning: Projected magnetisation not implemented for QE. Returning None.

[PARSED_WITH_WARNING] Folder: Mg_O_+2 | Warning: Projected magnetisation not implemented for QE. Returning None.

[PARSED_WITH_WARNING] Folder: Mg_O_+1 | Warning: P

# VASP

In [7]:
dp_vasp = DefectsParser(
   output_path="MgO/Defects/Pre_Calculated_Results",
   dielectric=dielectric,
   code = 'vasp'
, processes =1 )
dp_vasp_ENCUT_400 = DefectsParser(
   output_path="MgO/ENCUT_400_Defects/",  # both defect and bulk have ENCUT = 400, consistent with no warnings shown here
   dielectric=dielectric,
   code = 'vasp', processes = 1
)
vasp_defect_entry_dict = {**dp_vasp.defect_dict, **dp_vasp_ENCUT_400.defect_dict}

Parsed Mg_O_+3/vasp_std: 100%|██████████| 5/5 [00:20<00:00,  4.11s/it]            
Multiple `vasprun.xml` files found in certain defect directories:
(directory: chosen file for parsing):
/Users/atharvaanturkar/PycharmProjects/doped_QE/examples/MgO/Defects/Pre_Calculated_Results/Mg_O_+1/vasp_std: vasprun.xml.gz
/Users/atharvaanturkar/PycharmProjects/doped_QE/examples/MgO/Defects/Pre_Calculated_Results/Mg_O_0/vasp_std: vasprun.xml.gz
vasprun.xml files are used to parse the calculation energy and metadata.

Parsed Mg_O_Unrelaxed_+1/vasp_std: 100%|██████████| 1/1 [00:07<00:00,  7.43s/it]  


# QE-VASP comparison

In [10]:
import pandas as pd
from doped.utils.parsing import BOHR_TO_ANGSTROM

charges = [1,2,3,4]
keys = [ "Mg_O_+1", "Mg_O_+2", "Mg_O_+3", "Mg_O_+4"]
for beta, dp in beta_dp_dict.items():
    print(f"Beta: {beta} Bohr (= {beta*BOHR_TO_ANGSTROM:.2f} Å)")
    corrections_qe = [dp.defect_dict[key].get_kumagai_correction(verbose=False) for key in keys]
    corrections_vasp = [vasp_defect_entry_dict[key].get_kumagai_correction(verbose=False) for key in keys]

    df = pd.DataFrame({
        "q": charges, 
        "eFNV VASP:": [i.correction_energy for i in corrections_vasp], 
        "eFNV espresso:": [i.correction_energy for i in corrections_qe ]
    })
    df = df.set_index("q")
    df["ΔeFNV (eV)"] = abs(df["eFNV VASP:"] - df["eFNV espresso:"])
    df["ΔeFNV (%)"] = round(abs(df["ΔeFNV (eV)"] / df["eFNV VASP:"]) * 100, 1)
    for col in ["eFNV VASP:", "eFNV espresso:", "ΔeFNV (eV)"]:
        df[col] = round(df[col], 3)
    print(df)

Beta: 0.5 Bohr (= 0.26 Å)
   eFNV VASP:  eFNV QE:  ΔeFNV (eV)  ΔeFNV (%)
q                                             
1       0.199     0.195       0.004        2.1
2       0.723     0.725       0.001        0.2
3       1.572     1.572       0.001        0.1
4       2.586     2.601       0.015        0.6
Beta: 1.0 Bohr (= 0.53 Å)
   eFNV VASP:  eFNV QE:  ΔeFNV (eV)  ΔeFNV (%)
q                                             
1       0.199     0.173       0.025       12.7
2       0.723     0.681       0.042        5.8
3       1.572     1.510       0.062        3.9
4       2.586     2.517       0.069        2.7
Beta: 1.5 Bohr (= 0.79 Å)
   eFNV VASP:  eFNV QE:  ΔeFNV (eV)  ΔeFNV (%)
q                                             
1       0.199     0.144       0.055       27.7
2       0.723     0.623       0.100       13.8
3       1.572     1.418       0.153        9.7
4       2.586     2.397       0.189        7.3
Beta: 2.0 Bohr (= 1.06 Å)
   eFNV VASP:  eFNV QE:  ΔeFNV (eV)  ΔeFNV (%)
q  

Ok errors clearly getting significantly worse as we increase beta.

In [13]:

calc_root = "../tests/data/espresso/MgO_qe/Defects"
pp_folder = "../tests/data/espresso/pp_folder"

beta_dp_dict = {}
for beta in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]:
    beta_dp_dict[beta] = DefectsParser(code = 'espresso',
                   output_path = calc_root,
                   dielectric= 8.8963,
                   beta = beta,
                   pp_folder = pp_folder, bulk_path = bulk_path, processes = 1)

Parsing Mg_O_+1/espresso_std: 100%|██████████| 4/4 [00:38<00:00,  9.70s/it]
analysis.py:5357: UserWarning: [PARSED_WITH_WARNING] Folder: Mg_O_+3 | Warning: Projected magnetisation not implemented for QE. Returning None.

[PARSED_WITH_WARNING] Folder: Mg_O_+4 | Warning: Projected magnetisation not implemented for QE. Returning None.

[PARSED_WITH_WARNING] Folder: Mg_O_+2 | Warning: Projected magnetisation not implemented for QE. Returning None.

[PARSED_WITH_WARNING] Folder: Mg_O_+1 | Warning: Projected magnetisation not implemented for QE. Returning None.
Parsing Mg_O_+1/espresso_std: 100%|██████████| 4/4 [00:38<00:00,  9.70s/it]
analysis.py:5357: UserWarning: [PARSED_WITH_WARNING] Folder: Mg_O_+3 | Warning: Projected magnetisation not implemented for QE. Returning None.

[PARSED_WITH_WARNING] Folder: Mg_O_+4 | Warning: Projected magnetisation not implemented for QE. Returning None.

[PARSED_WITH_WARNING] Folder: Mg_O_+2 | Warning: Projected magnetisation not implemented for QE. Return

In [14]:
import pandas as pd
from doped.utils.parsing import BOHR_TO_ANGSTROM

charges = [1,2,3,4]
keys = ["Mg_O_+1", "Mg_O_+2", "Mg_O_+3", "Mg_O_+4"]
for beta, dp in beta_dp_dict.items():
    print(f"Beta: {beta} Bohr (= {beta*BOHR_TO_ANGSTROM:.2f} Å)")
    corrections_qe = [dp.defect_dict[key].get_kumagai_correction(verbose=False) for key in keys]
    corrections_vasp = [vasp_defect_entry_dict[key].get_kumagai_correction(verbose=False) for key in keys]

    df = pd.DataFrame({
        "q": charges,
        "eFNV VASP:": [i.correction_energy for i in corrections_vasp],
        "eFNV espresso:": [i.correction_energy for i in corrections_qe ]
    })
    df = df.set_index("q")
    df["ΔeFNV (eV)"] = abs(df["eFNV VASP:"] - df["eFNV espresso:"])
    df["ΔeFNV (%)"] = round(abs(df["ΔeFNV (eV)"] / df["eFNV VASP:"]) * 100, 1)
    for col in ["eFNV VASP:", "eFNV espresso:", "ΔeFNV (eV)"]:
        df[col] = round(df[col], 3)
    print(df)
    print(f"Average error: {df['ΔeFNV (eV)'].mean():.3f} eV, {df['ΔeFNV (%)'].mean():.1f}%\n")

Beta: 0.1 Bohr (= 0.05 Å)
   eFNV VASP:  eFNV QE:  ΔeFNV (eV)  ΔeFNV (%)
q                                             
1       0.199     0.234       0.035       17.8
2       0.723     0.749       0.026        3.6
3       1.572     1.658       0.087        5.5
4       2.586     2.622       0.036        1.4
Average error: 0.046 eV, 7.1%

Beta: 0.2 Bohr (= 0.11 Å)
   eFNV VASP:  eFNV QE:  ΔeFNV (eV)  ΔeFNV (%)
q                                             
1       0.199     0.214       0.015        7.6
2       0.723     0.737       0.014        2.0
3       1.572     1.610       0.038        2.4
4       2.586     2.607       0.021        0.8
Average error: 0.022 eV, 3.2%

Beta: 0.3 Bohr (= 0.16 Å)
   eFNV VASP:  eFNV QE:  ΔeFNV (eV)  ΔeFNV (%)
q                                             
1       0.199     0.201       0.002        1.0
2       0.723     0.732       0.009        1.2
3       1.572     1.582       0.010        0.7
4       2.586     2.604       0.018        0.7
Average error:

Best match is with 0.5 Bohr (0.26 Å) here.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


def _unify_data_limits_across_figures(*figs):
    """Set matching axis indices in each figure to the union of x/y limits."""
    axes_per_fig = [f.get_axes() for f in figs]
    if not axes_per_fig or any(not axs for axs in axes_per_fig):
        return
    n_ax = min(len(axs) for axs in axes_per_fig)
    for i in range(n_ax):
        x0 = min(axs[i].get_xlim()[0] for axs in axes_per_fig)
        x1 = max(axs[i].get_xlim()[1] for axs in axes_per_fig)
        y0 = min(axs[i].get_ylim()[0] for axs in axes_per_fig)
        y1 = max(axs[i].get_ylim()[1] for axs in axes_per_fig)
        for axs in axes_per_fig:
            axs[i].set_xlim(x0, x1)
            axs[i].set_ylim(y0, y1)


def show_side_by_side(fig_left, fig_right, figsize=(8, 5)):
    _unify_data_limits_across_figures(fig_left, fig_right)
    fig_right.axes[0].set_ylabel("")  # drop y-label and tick-labels for fig_right
    fig_right.axes[0].set_yticklabels([])
    fig, axs = plt.subplots(
        1, 2, figsize=figsize, 
        constrained_layout=True,
        gridspec_kw={'wspace': 0}  # decrease wspace to reduce horizontal gap
    )
    for ax, f in zip(axs, (fig_left, fig_right)):
        f.canvas.draw()
        w, h = f.canvas.get_width_height()
        rgb = np.frombuffer(f.canvas.buffer_rgba(), dtype=np.uint8).reshape(h, w, 4)[..., :3]
        ax.imshow(rgb)
        ax.axis("off")

    return fig


In [2]:
show_side_by_side(
    beta_dp_dict[0.5].defect_dict["Mg_O_Unperturbed_+1"].get_kumagai_correction(plot=True)[1], 
    vasp_defect_entry_dict["Mg_O_Unperturbed_+1"].get_kumagai_correction(plot=True)[1]
)

In [11]:
show_side_by_side(
    beta_dp_dict[0.5].defect_dict["Mg_O_+1"].get_kumagai_correction(plot=True)[1], 
    vasp_defect_entry_dict["Mg_O_+1"].get_kumagai_correction(plot=True)[1]
)

Parsing v_Sb_-3/espresso_std: 100%|██████████| 1/1 [00:21<00:00, 21.81s/it]
analysis.py:5102: UserWarning: [PARSED_WITH_WARNING] Folder: v_Sb_-3 | Warning: Projected magnetisation not implemented for QE. Returning None.
Static dielectric constant not implemented for QE. Returning None.
Static dielectric constant without local field effects not implemented for QE. Returning None.
Ionic part of the static dielectric constant not implemented for QE. Returning None.
CBM and band_gap are infinite, which can cause downstream failures. Reverting to use of bulk supercell calculation for band edge extrema.Add extra bands using the 'nbnd' parameter
analysis.py:4848: UserWarning: Estimated error in the Kumagai (eFNV) charge correction for certain defects is greater than the `error_tolerance` (= 0.050 eV):
v_Sb_-3: 0.055 eV
You may want to check the accuracy of the corrections by plotting the site potential differences (using `defect_entry.get_kumagai_correction()` with `plot=True`). Large errors 

In [5]:
show_side_by_side(
    beta_dp_dict[0.5].defect_dict["Mg_O_+2"].get_kumagai_correction(plot=True)[1], 
    vasp_defect_entry_dict["Mg_O_+2"].get_kumagai_correction(plot=True)[1]
)

In [ ]:
show_side_by_side(
    beta_dp_dict[0.5].defect_dict["Mg_O_+3"].get_kumagai_correction(plot=True)[1], 
    vasp_defect_entry_dict["Mg_O_+3"].get_kumagai_correction(plot=True)[1]
)

In [8]:
show_side_by_side(
    beta_dp_dict[0.5].defect_dict["Mg_O_+4"].get_kumagai_correction(plot=True)[1], 
    vasp_defect_entry_dict["Mg_O_+4"].get_kumagai_correction(plot=True)[1]
)

Parsing v_Sb_-3/espresso_std: 100%|██████████| 1/1 [00:20<00:00, 20.09s/it]
analysis.py:5102: UserWarning: [PARSED_WITH_WARNING] Folder: v_Sb_-3 | Warning: Projected magnetisation not implemented for QE. Returning None.
Static dielectric constant not implemented for QE. Returning None.
Static dielectric constant without local field effects not implemented for QE. Returning None.
Ionic part of the static dielectric constant not implemented for QE. Returning None.
CBM and band_gap are infinite, which can cause downstream failures. Reverting to use of bulk supercell calculation for band edge extrema.Add extra bands using the 'nbnd' parameter
analysis.py:4848: UserWarning: Estimated error in the Kumagai (eFNV) charge correction for certain defects is greater than the `error_tolerance` (= 0.050 eV):
v_Sb_-3: 0.055 eV
You may want to check the accuracy of the corrections by plotting the site potential differences (using `defect_entry.get_kumagai_correction()` with `plot=True`). Large errors 

# Element-Wise Comparisons

In [ ]:
import pandas as pd
from doped.utils.parsing import BOHR_TO_ANGSTROM

keys = ["Mg_O_Unperturbed_+1", "Mg_O_+1", "Mg_O_+2", "Mg_O_+3", "Mg_O_+4"]
for beta, dp in beta_dp_dict.items():
    print(f"Beta: {beta} Bohr (= {beta*BOHR_TO_ANGSTROM:.2f} Å)")
    corrections_qe = [dp.defect_dict[key].get_kumagai_correction(verbose=False) for key in keys]
    corrections_vasp = [vasp_defect_entry_dict[key].get_kumagai_correction(verbose=False) for key in keys]

    # get average potential diff for O/Mg sites
    def get_avg_potential_diff(correction1, correction2, specie):
        # relies on matching ordering btw
        correction1_potential_diffs = np.array([
            correction1.metadata["pydefect_ExtendedFnvCorrection"].sites[i].potential
            for i in range(len(correction1.metadata["pydefect_ExtendedFnvCorrection"].sites))
            if correction1.metadata["pydefect_ExtendedFnvCorrection"].sites[i].specie == specie
        ])
        correction2_potential_diffs = np.array([
            correction2.metadata["pydefect_ExtendedFnvCorrection"].sites[i].potential
            for i in range(len(correction2.metadata["pydefect_ExtendedFnvCorrection"].sites))
            if correction2.metadata["pydefect_ExtendedFnvCorrection"].sites[i].specie == specie
        ])
        return np.mean(np.abs(correction1_potential_diffs - correction2_potential_diffs))

    avg_diff_O = [get_avg_potential_diff(corrections_qe[i], corrections_vasp[i], "O") for i in range(len(keys))]
    avg_diff_Mg = [get_avg_potential_diff(corrections_qe[i], corrections_vasp[i], "Mg") for i in range(len(keys))]

    df = pd.DataFrame({
        "q": [1,1,2,3,4], 
        "eFNV VASP:": [i.correction_energy for i in corrections_vasp], 
        "eFNV espresso:": [i.correction_energy for i in corrections_qe ]
    })
    df = df.set_index("q")
    df["ΔeFNV (eV)"] = abs(df["eFNV VASP:"] - df["eFNV espresso:"])
    df["ΔeFNV (%)"] = round(abs(df["ΔeFNV (eV)"] / df["eFNV VASP:"]) * 100, 1)
    df["Δ|V| (O); eV"] = avg_diff_O
    df["Δ|V| (Mg); eV"] = avg_diff_Mg
    for col in ["eFNV VASP:", "eFNV espresso:", "ΔeFNV (eV)", "Δ|V| (O); eV", "Δ|V| (Mg); eV"]:
        df[col] = round(df[col], 3)
    print(df)
    print(f"Average error: {df['ΔeFNV (eV)'].mean():.3f} eV, {df['ΔeFNV (%)'].mean():.1f}%")
    print(f"Average potential diff (O): {np.mean(avg_diff_O):.3f} eV")
    print(f"Average potential diff (Mg): {np.mean(avg_diff_Mg):.3f} eV")
    print("")

Zoom in to ~0.5 Bohr beta values:

In [ ]:
calc_root = "../tests/data/espresso/MgO_qe"
pp_folder = "../tests/data/espresso/pp_folder"

for beta in [0.45, 0.55]:
    beta_dp_dict[beta] = DefectsParser(code = 'espresso',
                   output_path = calc_root,
                   dielectric= 8.8963,
                   beta = beta,
                   pp_folder = pp_folder, bulk_path = bulk_path)

In [ ]:
import pandas as pd
from doped.utils.parsing import BOHR_TO_ANGSTROM

keys = ["Mg_O_Unperturbed_+1", "Mg_O_+1", "Mg_O_+2", "Mg_O_+3", "Mg_O_+4"]
for beta, dp in beta_dp_dict.items():
    if beta < 0.4 or beta > 0.55:
        continue
    print(f"Beta: {beta} Bohr (= {beta*BOHR_TO_ANGSTROM:.2f} Å)")
    corrections_qe = [dp.defect_dict[key].get_kumagai_correction(verbose=False) for key in keys]
    corrections_vasp = [vasp_defect_entry_dict[key].get_kumagai_correction(verbose=False) for key in keys]

    # get average potential diff for O/Mg sites
    def get_avg_potential_diff(correction1, correction2, specie):
        # relies on matching ordering btw
        correction1_potential_diffs = np.array([
            correction1.metadata["pydefect_ExtendedFnvCorrection"].sites[i].potential
            for i in range(len(correction1.metadata["pydefect_ExtendedFnvCorrection"].sites))
            if correction1.metadata["pydefect_ExtendedFnvCorrection"].sites[i].specie == specie
        ])
        correction2_potential_diffs = np.array([
            correction2.metadata["pydefect_ExtendedFnvCorrection"].sites[i].potential
            for i in range(len(correction2.metadata["pydefect_ExtendedFnvCorrection"].sites))
            if correction2.metadata["pydefect_ExtendedFnvCorrection"].sites[i].specie == specie
        ])
        return np.mean(np.abs(correction1_potential_diffs - correction2_potential_diffs))

    avg_diff_O = [get_avg_potential_diff(corrections_qe[i], corrections_vasp[i], "O") for i in range(len(keys))]
    avg_diff_Mg = [get_avg_potential_diff(corrections_qe[i], corrections_vasp[i], "Mg") for i in range(len(keys))]

    df = pd.DataFrame({
        "q": [1,1,2,3,4], 
        "eFNV VASP:": [i.correction_energy for i in corrections_vasp], 
        "eFNV espresso:": [i.correction_energy for i in corrections_qe ]
    })
    df = df.set_index("q")
    df["ΔeFNV (eV)"] = abs(df["eFNV VASP:"] - df["eFNV espresso:"])
    df["ΔeFNV (%)"] = round(abs(df["ΔeFNV (eV)"] / df["eFNV VASP:"]) * 100, 1)
    df["Δ|V| (O); eV"] = avg_diff_O
    df["Δ|V| (Mg); eV"] = avg_diff_Mg
    for col in ["eFNV VASP:", "eFNV espresso:", "ΔeFNV (eV)", "Δ|V| (O); eV", "Δ|V| (Mg); eV"]:
        df[col] = round(df[col], 3)
    print(df)
    print(f"Average error: {df['ΔeFNV (eV)'].mean():.3f} eV, {df['ΔeFNV (%)'].mean():.1f}%")
    print(f"Average potential diff (O): {np.mean(avg_diff_O):.3f} eV")
    print(f"Average potential diff (Mg): {np.mean(avg_diff_Mg):.3f} eV")
    print("")

The optimal beta to minimise site potential discrepancies between QE and VASP here is mostly consistent between the two elements, suggesting no major benefit to using element-specific betas.

# $Sb_{2}Si_{2}Te_{6}$ Beta Tests

# QE

In [ ]:
from pymatgen.io.vasp import VolumetricData
from doped.analysis import DefectsParser
%matplotlib inline

In [ ]:
from pymatgen.io.vasp import VolumetricData
from doped.analysis import DefectsParser
%matplotlib inline
import numpy as np
calc_root_aniso = "../tests/data/espresso/Sb2Si2Te6_QE"
pp_folder = "../tests/data/espresso/pp_folder"
dielectric_aniso = np.array([44.12,44.12, 17.82])


In [ ]:

beta_dp_dict = {}
for beta in [0.5, 1.0, 1.5, 2.0]:
    beta_dp_dict[beta] = DefectsParser(code = 'espresso',
                   output_path = calc_root_aniso,
                   dielectric= dielectric_aniso,
                   beta = beta,
                   pp_folder = pp_folder, processes = 1)

# VASP

In [ ]:
calc_root_aniso_vasp = "Sb2Si2Te6"

In [ ]:
dp_aniso_vasp = DefectsParser(code = 'vasp',
                   output_path = calc_root_aniso_vasp,
                   dielectric=dielectric_aniso)

# QE-VASP Comparison

In [ ]:
import pandas as pd
from doped.utils.parsing import BOHR_TO_ANGSTROM

charges = [-3]
keys = ["v_Sb_-3"]
for beta, dp_aniso in beta_dp_dict.items():
    print(f"Beta: {beta} Bohr (= {beta*BOHR_TO_ANGSTROM:.2f} Å)")
    corrections_qe = [dp_aniso.defect_dict[key].get_kumagai_correction(verbose=False) for key in keys]
    corrections_vasp = [dp_aniso_vasp.defect_dict[key].get_kumagai_correction(verbose=False) for key in keys]

    df = pd.DataFrame({
        "q": charges,
        "eFNV VASP:": [i.correction_energy for i in corrections_vasp],
        "eFNV espresso:": [i.correction_energy for i in corrections_qe ]
    })
    df = df.set_index("q")
    df["ΔeFNV (eV)"] = abs(df["eFNV VASP:"] - df["eFNV espresso:"])
    df["ΔeFNV (%)"] = round(abs(df["ΔeFNV (eV)"] / df["eFNV VASP:"]) * 100, 1)
    for col in ["eFNV VASP:", "eFNV espresso:", "ΔeFNV (eV)"]:
        df[col] = round(df[col], 3)
    print(df)

In [ ]:

beta_dp_dict = {}
for beta in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]:
    beta_dp_dict[beta] = DefectsParser(code = 'espresso',
                   output_path = calc_root_aniso,
                   dielectric= dielectric_aniso,
                   beta = beta,
                   pp_folder = pp_folder, processes = 1)

In [ ]:
import pandas as pd

charges = [-3]
keys = ["v_Sb_-3"]
for beta, dp in beta_dp_dict.items():
    print(f"Beta: {beta} Bohr (= {beta*BOHR_TO_ANGSTROM:.2f} Å)")
    corrections_qe = [dp.defect_dict[key].get_kumagai_correction(verbose=False) for key in keys]
    corrections_vasp = [dp_aniso_vasp.defect_dict[key].get_kumagai_correction(verbose=False) for key in keys]

    df = pd.DataFrame({
        "q": charges,
        "eFNV VASP:": [i.correction_energy for i in corrections_vasp],
        "eFNV espresso:": [i.correction_energy for i in corrections_qe ]
    })
    df = df.set_index("q")
    df["ΔeFNV (eV)"] = abs(df["eFNV VASP:"] - df["eFNV espresso:"])
    df["ΔeFNV (%)"] = round(abs(df["ΔeFNV (eV)"] / df["eFNV VASP:"]) * 100, 1)
    for col in ["eFNV VASP:", "eFNV espresso:", "ΔeFNV (eV)"]:
        df[col] = round(df[col], 3)
    print(df)
    print(f"Average error: {df['ΔeFNV (eV)'].mean():.3f} eV, {df['ΔeFNV (%)'].mean():.1f}%\n")